# Notebook 3 — Explicación de la API y el Microservicio
## Proyecto EpiDiagnostix-Mayab — Componente ML

**Para:** Estudio personal y defensa ante el profesor  
**Qué cubre:** Explica qué es cada pieza del microservicio, por qué existe y cómo se conectan entre sí.

> Este notebook **no ejecuta la API** — la explica conceptualmente con código de ejemplo.

---
# PARTE 1 — ¿Qué es una API REST y FastAPI?

## API REST

Una **API** (Application Programming Interface) es un contrato entre dos programas:
- *Uno pregunta* (el cliente — la app móvil)
- *Otro responde* (el servidor — nuestro microservicio)

**REST** define cómo se hacen esas preguntas: usando el protocolo HTTP con verbos estándar.

| Verbo HTTP | Para qué | Nuestros endpoints |
|---|---|---|
| `GET` | Consultar datos | `/health`, `/inferencias` |
| `POST` | Enviar datos para procesamiento | `/extraccion`, `/deteccion-anomalias`, `/consulta-completa` |

## FastAPI

FastAPI es el framework de Python que usamos. Tiene tres ventajas clave:
1. **Validación automática** con Pydantic (si mandas `edad: "hola"`, devuelve error 422 sin que tú lo programes)
2. **Swagger/OpenAPI** generado automáticamente en `/docs` (documentación interactiva)
3. **Muy rápido** — basado en Starlette y estándares ASGI

In [1]:
# Demostración conceptual del flujo de una petición
print('Flujo de una petición POST /consulta-completa:')
print()
print('  [App Móvil]')
print('      │')
print('      │ HTTP POST /consulta-completa')
print('      │ Body: {"texto": "Paciente masculino de 45 años..."}')
print('      ▼')
print('  [FastAPI — infrastructure/api/routers/completa.py]')
print('      │')
print('      │ Pydantic valida que texto.length >= 10')
print('      │ Si no pasa: devuelve 422 Unprocessable Entity')
print('      │ Si pasa: llama al caso de uso')
print('      ▼')
print('  [ConsultaCompletaUseCase — application/use_cases.py]')
print('      │')
print('      ├──► ExtractorAdapter.extraer(texto)     → ResultadoExtraccion')
print('      │')
print('      ├──► IsolationForestAdapter.detectar(consulta) → ResultadoAnomalia')
print('      │')
print('      └──► PostgresRepository.guardar(inferencia)')
print('      ▼')
print('  [FastAPI serializa la respuesta con Pydantic]')
print('      │')
print('      │ HTTP 200 OK')
print('      │ Body: {"inferencia_id": "...", "extraccion": {...}, "anomalia": {...}}')
print('      ▼')
print('  [App Móvil recibe la respuesta]')

Flujo de una petición POST /consulta-completa:

  [App Móvil]
      │
      │ HTTP POST /consulta-completa
      │ Body: {"texto": "Paciente masculino de 45 años..."}
      ▼
  [FastAPI — infrastructure/api/routers/completa.py]
      │
      │ Pydantic valida que texto.length >= 10
      │ Si no pasa: devuelve 422 Unprocessable Entity
      │ Si pasa: llama al caso de uso
      ▼
  [ConsultaCompletaUseCase — application/use_cases.py]
      │
      ├──► ExtractorAdapter.extraer(texto)     → ResultadoExtraccion
      │
      ├──► IsolationForestAdapter.detectar(consulta) → ResultadoAnomalia
      │
      └──► PostgresRepository.guardar(inferencia)
      ▼
  [FastAPI serializa la respuesta con Pydantic]
      │
      │ HTTP 200 OK
      │ Body: {"inferencia_id": "...", "extraccion": {...}, "anomalia": {...}}
      ▼
  [App Móvil recibe la respuesta]


---
# PARTE 2 — Arquitectura Hexagonal (Ports & Adapters)

## ¿Por qué arquitectura hexagonal?

**El problema** que resuelve: los modelos de ML están acoplados a scikit-learn, spaCy, PostgreSQL. Si mañana quieres cambiar PostgreSQL por MongoDB, o sklearn por otro framework, ¿tienes que reescribir toda la aplicación?

Con arquitectura hexagonal: **NO**. Solo cambias el adapter correspondiente.

## Concepto de Puerto (Port) y Adapter

Un **Puerto** es un contrato (interfaz abstracta) que define qué puede hacer un componente, sin decir cómo.

Un **Adapter** es la implementación concreta de ese contrato para una tecnología específica.

In [2]:
# Demostración: Port vs Adapter
from abc import ABC, abstractmethod

# El PUERTO — domain/ports.py
class AnomalyDetectorPort(ABC):
    """Contrato: cualquier cosa que quiera ser un detector de anomalías
       debe implementar este método. No importa cómo lo haga."""
    @abstractmethod
    def detectar(self, consulta) -> dict:
        ...


# ADAPTER real — infrastructure/adapters/isolation_forest_adapter.py
class IsolationForestAdapter(AnomalyDetectorPort):
    """Implementación concreta usando scikit-learn."""
    def detectar(self, consulta) -> dict:
        # Aquí carga el modelo joblib, escala los datos, hace predict...
        return {"es_anomalia": False, "score": 0.1}


# ADAPTER alternativo hipotético — si mañana cambiamos a PyOD
class PyODAdapter(AnomalyDetectorPort):
    """Implementación alternativa usando PyOD — mismo puerto, distinta tecnología."""
    def detectar(self, consulta) -> dict:
        # ... lógica con PyOD ...
        return {"es_anomalia": False, "score": 0.08}


# El caso de uso NO sabe ni le importa cuál adapter está usando
def ejecutar_caso_uso(detector: AnomalyDetectorPort, consulta):
    resultado = detector.detectar(consulta)
    return resultado


# Con IsolationForest:
print('Con IsolationForest:', ejecutar_caso_uso(IsolationForestAdapter(), None))
# Con PyOD (hipotético):
print('Con PyOD:           ', ejecutar_caso_uso(PyODAdapter(), None))
print()
print('El caso de uso es IDÉNTICO — solo cambia el adapter.')

Con IsolationForest: {'es_anomalia': False, 'score': 0.1}
Con PyOD:            {'es_anomalia': False, 'score': 0.08}

El caso de uso es IDÉNTICO — solo cambia el adapter.


In [3]:
# Diagrama de capas de la arquitectura
print()
print('DIAGRAMA — Arquitectura Hexagonal EpiDiagnostix-Mayab ML')
print()
print('  ┌─────────────────────────────────────────────────────────┐')
print('  │                   INFRASTRUCTURE                        │')
print('  │  ┌──────────┐  ┌──────────────┐  ┌──────────────────┐  │')
print('  │  │ FastAPI   │  │ ExtractorAd. │  │ PostgresRepo.    │  │')
print('  │  │ (entrada) │  │ (regex/spaCy)│  │ (SQLAlchemy)     │  │')
print('  │  │ routers/  │  │              │  │                  │  │')
print('  │  │ schemas/  │  │ IsolationFor │  │                  │  │')
print('  │  └─────┬─────┘  │ estAdapter   │  └────────┬─────────┘  │')
print('  │        │        └──────┬───────┘           │            │')
print('  └────────│───────────────│───────────────────│────────────┘')
print('           │               │                   │')
print('  ┌────────▼───────────────▼───────────────────▼────────────┐')
print('  │                   APPLICATION                           │')
print('  │  ExtraerVariablesUseCase | DetectarAnomaliaUseCase     │')
print('  │  ConsultaCompletaUseCase | ConsultarHistorialUseCase   │')
print('  │                   (Ports como interfaces)               │')
print('  └────────────────────────┬────────────────────────────────┘')
print('                           │')
print('  ┌────────────────────────▼────────────────────────────────┐')
print('  │                     DOMAIN                              │')
print('  │  ConsultaClinica | ResultadoExtraccion | Inferencia     │')
print('  │  ExtractorPort | AnomalyDetectorPort                   │')
print('  │       SIN dependencias externas — Python puro           │')
print('  └─────────────────────────────────────────────────────────┘')


DIAGRAMA — Arquitectura Hexagonal EpiDiagnostix-Mayab ML

  ┌─────────────────────────────────────────────────────────┐
  │                   INFRASTRUCTURE                        │
  │  ┌──────────┐  ┌──────────────┐  ┌──────────────────┐  │
  │  │ FastAPI   │  │ ExtractorAd. │  │ PostgresRepo.    │  │
  │  │ (entrada) │  │ (regex/spaCy)│  │ (SQLAlchemy)     │  │
  │  │ routers/  │  │              │  │                  │  │
  │  │ schemas/  │  │ IsolationFor │  │                  │  │
  │  └─────┬─────┘  │ estAdapter   │  └────────┬─────────┘  │
  │        │        └──────┬───────┘           │            │
  └────────│───────────────│───────────────────│────────────┘
           │               │                   │
  ┌────────▼───────────────▼───────────────────▼────────────┐
  │                   APPLICATION                           │
  │  ExtraerVariablesUseCase | DetectarAnomaliaUseCase     │
  │  ConsultaCompletaUseCase | ConsultarHistorialUseCase   │
  │                   (Port

---
# PARTE 3 — Pydantic: Validación de entrada y salida

Pydantic es la librería que valida automáticamente los datos que llegan a la API.

In [4]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional

# Esquema de validación para el endpoint /deteccion-anomalias
class DeteccionRequest(BaseModel):
    edad:                    int   = Field(..., ge=0, le=120)
    glucosa_mg_dl:           int   = Field(..., ge=20, le=800)
    temperatura_c:           float = Field(..., ge=30, le=45)
    categoria_sintoma:       str

# Caso 1: datos válidos
print('Caso 1 — Datos válidos:')
try:
    req = DeteccionRequest(edad=45, glucosa_mg_dl=98, temperatura_c=37.2, categoria_sintoma="Respiratorio")
    print(f'  OK: {req}')
except ValidationError as e:
    print(f'  Error: {e}')

print()
print('Caso 2 — Edad inválida (200 años):')
try:
    req = DeteccionRequest(edad=200, glucosa_mg_dl=98, temperatura_c=37.2, categoria_sintoma="Respiratorio")
    print(f'  OK: {req}')
except ValidationError as e:
    print(f'  Error capturado por Pydantic: {e.errors()[0]["msg"]}')

print()
print('Caso 3 — Tipo incorrecto (edad como string):')
try:
    req = DeteccionRequest(edad="hola", glucosa_mg_dl=98, temperatura_c=37.2, categoria_sintoma="X")
    print(f'  OK: {req}')
except ValidationError as e:
    print(f'  Error capturado por Pydantic: {e.errors()[0]["msg"]}')

Caso 1 — Datos válidos:
  OK: edad=45 glucosa_mg_dl=98 temperatura_c=37.2 categoria_sintoma='Respiratorio'

Caso 2 — Edad inválida (200 años):
  Error capturado por Pydantic: Input should be less than or equal to 120

Caso 3 — Tipo incorrecto (edad como string):
  Error capturado por Pydantic: Input should be a valid integer, unable to parse string as an integer


---
# PARTE 4 — ¿Qué hace cada endpoint?

## Endpoint 1: `GET /health`

El más simple — solo confirma que el contenedor está vivo. Docker lo llama automáticamente cada 10 segundos para saber si el servicio sigue funcionando.

In [5]:
import json

# Respuesta del /health
health_response = {
    "status": "ok",
    "version": "1.0.0",
    "modelos": {
        "isolation_forest": "isolation_forest.joblib",
        "extractor_ner":    "extractor_ner_config.joblib",
        "scaler":           "scaler_if.joblib"
    }
}
print('GET /health → HTTP 200')
print(json.dumps(health_response, indent=2))

GET /health → HTTP 200
{
  "status": "ok",
  "version": "1.0.0",
  "modelos": {
    "isolation_forest": "isolation_forest.joblib",
    "extractor_ner": "extractor_ner_config.joblib",
    "scaler": "scaler_if.joblib"
  }
}


## Endpoint 2: `POST /extraccion`

Recibe texto libre → aplica el extractor NER (regex + spaCy) → devuelve variables estructuradas.

In [6]:
# Simulación del flujo interno del endpoint /extraccion
import sys
sys.path.insert(0, '..')

import re, unicodedata

texto_entrada = "Paciente masculino de 45 años. Presión 125/82 mmHg, glucosa 98 mg/dl, temperatura 37.2°C."

print('POST /extraccion')
print(f'  Request: {{"texto": "{texto_entrada[:60]}..."}}')
print()
print('  Internamente llama a:')
print('  1. ExtractorAdapter.extraer(texto)')
print('     └─ Aplica 7 regex diferentes para cada campo')
print('  2. PostgresRepository.guardar(inferencia)')
print('     └─ INSERT INTO inferencias (tipo="extraccion", input_json, output_json)')
print()

response_extraccion = {
    "inferencia_id": "3fa85f64-5717-4562-b3fc-2c963f66afa6",
    "campos_extraidos": {
        "edad": 45, "sexo": "M",
        "presion_sistolica": 125, "presion_diastolica": 82,
        "glucosa_mg_dl": 98, "temperatura_c": 37.2,
        "peso_kg": None, "talla_cm": None,
        "frecuencia_cardiaca_bpm": None, "duracion_sintomas_dias": None,
        "categoria_sintoma": None
    },
    "campos_no_extraidos": ["peso_kg", "talla_cm", "frecuencia_cardiaca_bpm",
                             "duracion_sintomas_dias", "categoria_sintoma"],
    "created_at": "2025-11-01T10:00:00"
}
print('  Response HTTP 200:')
print(json.dumps(response_extraccion, indent=2, ensure_ascii=False))

POST /extraccion
  Request: {"texto": "Paciente masculino de 45 años. Presión 125/82 mmHg, glucosa ..."}

  Internamente llama a:
  1. ExtractorAdapter.extraer(texto)
     └─ Aplica 7 regex diferentes para cada campo
  2. PostgresRepository.guardar(inferencia)
     └─ INSERT INTO inferencias (tipo="extraccion", input_json, output_json)

  Response HTTP 200:
{
  "inferencia_id": "3fa85f64-5717-4562-b3fc-2c963f66afa6",
  "campos_extraidos": {
    "edad": 45,
    "sexo": "M",
    "presion_sistolica": 125,
    "presion_diastolica": 82,
    "glucosa_mg_dl": 98,
    "temperatura_c": 37.2,
    "peso_kg": null,
    "talla_cm": null,
    "frecuencia_cardiaca_bpm": null,
    "duracion_sintomas_dias": null,
    "categoria_sintoma": null
  },
  "campos_no_extraidos": [
    "peso_kg",
    "talla_cm",
    "frecuencia_cardiaca_bpm",
    "duracion_sintomas_dias",
    "categoria_sintoma"
  ],
  "created_at": "2025-11-01T10:00:00"
}


## Endpoint 3: `POST /deteccion-anomalias`

Recibe las variables ya estructuradas → aplica el Isolation Forest → devuelve si es anomalía.

**Nota:** Este endpoint asume que ya tienes las variables extraídas (por `/extraccion` o enviadas directo).

In [7]:
print('POST /deteccion-anomalias')
print()
print('  Internamente llama a:')
print('  1. IsolationForestAdapter.detectar(consulta)')
print('     ├─ Encode categoria_sintoma con LabelEncoder guardado')
print('     ├─ Construye vector de 10 features')
print('     ├─ scaler.transform(vector)   ← usa la media/std del entrenamiento')
print('     ├─ modelo.predict(vector_scaled)  → -1 (anomalía) o 1 (normal)')
print('     └─ modelo.decision_function()     → anomaly score')
print('  2. PostgresRepository.guardar(inferencia)')
print()
print('  Lógica de nivel_riesgo:')
print('    score >= 0       → nivel = "normal"')
print('    -0.15 < score < -0.05 → nivel = "sospechoso"')
print('    score <= -0.15   → nivel = "anomalo"')
print()

# Ejemplo paciente normal
r_normal = {
    "inferencia_id": "aab12345-...",
    "es_anomalia": False,
    "score": 0.042,
    "nivel_riesgo": "normal",
    "created_at": "2025-11-01T10:00:01"
}
# Ejemplo paciente anómalo (glucosa 341)
r_anomalo = {
    "inferencia_id": "cc99abcd-...",
    "es_anomalia": True,
    "score": -0.31,
    "nivel_riesgo": "anomalo",
    "created_at": "2025-11-01T10:00:02"
}

print('  Paciente normal (glucosa=87, temp=37.0):')
print(json.dumps(r_normal, indent=4))
print()
print('  Paciente anómalo (glucosa=341, temp=39.0):')
print(json.dumps(r_anomalo, indent=4))

POST /deteccion-anomalias

  Internamente llama a:
  1. IsolationForestAdapter.detectar(consulta)
     ├─ Encode categoria_sintoma con LabelEncoder guardado
     ├─ Construye vector de 10 features
     ├─ scaler.transform(vector)   ← usa la media/std del entrenamiento
     ├─ modelo.predict(vector_scaled)  → -1 (anomalía) o 1 (normal)
     └─ modelo.decision_function()     → anomaly score
  2. PostgresRepository.guardar(inferencia)

  Lógica de nivel_riesgo:
    score >= 0       → nivel = "normal"
    -0.15 < score < -0.05 → nivel = "sospechoso"
    score <= -0.15   → nivel = "anomalo"

  Paciente normal (glucosa=87, temp=37.0):
{
    "inferencia_id": "aab12345-...",
    "es_anomalia": false,
    "score": 0.042,
    "nivel_riesgo": "normal",
    "created_at": "2025-11-01T10:00:01"
}

  Paciente anómalo (glucosa=341, temp=39.0):
{
    "inferencia_id": "cc99abcd-...",
    "es_anomalia": true,
    "score": -0.31,
    "nivel_riesgo": "anomalo",
    "created_at": "2025-11-01T10:00:02"
}


## Endpoint 4: `POST /consulta-completa` — El principal

Este es el que usará la **app móvil** en campo. Encadena los dos pasos en una sola llamada.

In [8]:
print('POST /consulta-completa — Flujo interno')
print()
print('  Texto libre entra → ConsultaCompletaUseCase.ejecutar(texto)')
print()
print('  Paso 1: extractor.extraer(texto)')
print('    → devuelve campos_extraidos + campos_no_extraidos')
print()
print('  Paso 2: verificar campos requeridos para el IF:')
print('    [edad, peso_kg, talla_cm, presion_sistolica, presion_diastolica,')
print('     glucosa_mg_dl, temperatura_c, frecuencia_cardiaca_bpm,')
print('     duracion_sintomas_dias, categoria_sintoma]')
print()
print('  Si TODOS están presentes → detector.detectar(consulta)')
print('  Si FALTAN campos         → anomalia = None, advertencia = "Faltan campos [...]"')
print()
print('  Paso 3: repo.guardar(inferencia tipo="completa")')
print()

r_completa = {
    "inferencia_id": "fe12-...",
    "extraccion": {
        "edad": 45, "sexo": "M", "peso_kg": 82.0, "talla_cm": 170.5,
        "presion_sistolica": 125, "presion_diastolica": 82,
        "glucosa_mg_dl": 98, "temperatura_c": 37.2,
        "frecuencia_cardiaca_bpm": 91, "duracion_sintomas_dias": 5,
        "categoria_sintoma": "Gastrointestinal"
    },
    "anomalia": {
        "es_anomalia": False,
        "score": 0.042,
        "nivel_riesgo": "normal"
    },
    "campos_no_extraidos": [],
    "advertencia": None,
    "created_at": "2025-11-01T10:00:03"
}
print('  Response HTTP 200:')
print(json.dumps(r_completa, indent=2, ensure_ascii=False))

POST /consulta-completa — Flujo interno

  Texto libre entra → ConsultaCompletaUseCase.ejecutar(texto)

  Paso 1: extractor.extraer(texto)
    → devuelve campos_extraidos + campos_no_extraidos

  Paso 2: verificar campos requeridos para el IF:
    [edad, peso_kg, talla_cm, presion_sistolica, presion_diastolica,
     glucosa_mg_dl, temperatura_c, frecuencia_cardiaca_bpm,
     duracion_sintomas_dias, categoria_sintoma]

  Si TODOS están presentes → detector.detectar(consulta)
  Si FALTAN campos         → anomalia = None, advertencia = "Faltan campos [...]"

  Paso 3: repo.guardar(inferencia tipo="completa")

  Response HTTP 200:
{
  "inferencia_id": "fe12-...",
  "extraccion": {
    "edad": 45,
    "sexo": "M",
    "peso_kg": 82.0,
    "talla_cm": 170.5,
    "presion_sistolica": 125,
    "presion_diastolica": 82,
    "glucosa_mg_dl": 98,
    "temperatura_c": 37.2,
    "frecuencia_cardiaca_bpm": 91,
    "duracion_sintomas_dias": 5,
    "categoria_sintoma": "Gastrointestinal"
  },
  "anoma

## Endpoint 5: `GET /inferencias`

Historial de todas las inferencias guardadas en PostgreSQL, con filtros.

In [9]:
print('GET /inferencias?tipo=completa&limit=5')
print()
print('  Internamente: SELECT * FROM inferencias')
print('                WHERE tipo = "completa"')
print('                ORDER BY created_at DESC')
print('                LIMIT 5 OFFSET 0')
print()

r_historial = {
    "total": 42,
    "inferencias": [
        {"id": "fe12-...", "tipo": "completa", "score": 0.042,
         "es_anomalia": False, "created_at": "2025-11-01T10:00:03", "output_json": {}},
        {"id": "cc99-...", "tipo": "completa", "score": -0.31,
         "es_anomalia": True,  "created_at": "2025-11-01T09:55:00", "output_json": {}},
    ]
}
print('  Response HTTP 200:')
print(json.dumps(r_historial, indent=2))

GET /inferencias?tipo=completa&limit=5

  Internamente: SELECT * FROM inferencias
                WHERE tipo = "completa"
                ORDER BY created_at DESC
                LIMIT 5 OFFSET 0

  Response HTTP 200:
{
  "total": 42,
  "inferencias": [
    {
      "id": "fe12-...",
      "tipo": "completa",
      "score": 0.042,
      "es_anomalia": false,
      "created_at": "2025-11-01T10:00:03",
      "output_json": {}
    },
    {
      "id": "cc99-...",
      "tipo": "completa",
      "score": -0.31,
      "es_anomalia": true,
      "created_at": "2025-11-01T09:55:00",
      "output_json": {}
    }
  ]
}


---
# PARTE 5 — SQLAlchemy y PostgreSQL

## ¿Por qué PostgreSQL y no SQLite?

| | SQLite | PostgreSQL |
|---|---|---|
| Ideal para | Prototipo local, una sola persona | Producción, múltiples usuarios concurrentes |
| Tipo de archivo | Un archivo `.db` | Servidor independiente |
| Soporte JSON | Limitado | JSONB — binario indexado, consultas rápidas |
| Docker | No necesita contenedor extra | Requiere contenedor `postgres:16-alpine` |
| Rúbrica | No cumple (pide Postgres) | ✅ Cumple |

## ¿Qué es SQLAlchemy?

SQLAlchemy es un ORM (Object-Relational Mapper): traduce entre objetos Python y filas de base de datos.

In [10]:
# Así se ve la tabla en SQL puro
sql_create = """
CREATE TABLE inferencias (
    id          UUID        PRIMARY KEY DEFAULT gen_random_uuid(),
    tipo        VARCHAR(20) NOT NULL CHECK (tipo IN ('extraccion','anomalia','completa')),
    input_json  JSONB       NOT NULL,
    output_json JSONB       NOT NULL,
    score       FLOAT,           -- NULL cuando tipo = 'extraccion'
    es_anomalia BOOLEAN,         -- NULL cuando tipo = 'extraccion'
    created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
);
"""

# Y así se ve en SQLAlchemy (infrastructure/db/models.py)
python_orm = """
class InferenciaORM(Base):
    __tablename__ = "inferencias"
    
    id          = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    tipo        = Column(String(20), nullable=False)
    input_json  = Column(JSONB, nullable=False)
    output_json = Column(JSONB, nullable=False)
    score       = Column(Float, nullable=True)
    es_anomalia = Column(Boolean, nullable=True)
    created_at  = Column(DateTime, nullable=False, default=datetime.utcnow)
"""

print('SQL equivalente:')
print(sql_create)
print('ORM SQLAlchemy (lo que escribimos en Python):')
print(python_orm)

SQL equivalente:

CREATE TABLE inferencias (
    id          UUID        PRIMARY KEY DEFAULT gen_random_uuid(),
    tipo        VARCHAR(20) NOT NULL CHECK (tipo IN ('extraccion','anomalia','completa')),
    input_json  JSONB       NOT NULL,
    output_json JSONB       NOT NULL,
    score       FLOAT,           -- NULL cuando tipo = 'extraccion'
    es_anomalia BOOLEAN,         -- NULL cuando tipo = 'extraccion'
    created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
);

ORM SQLAlchemy (lo que escribimos en Python):

class InferenciaORM(Base):
    __tablename__ = "inferencias"
    
    id          = Column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    tipo        = Column(String(20), nullable=False)
    input_json  = Column(JSONB, nullable=False)
    output_json = Column(JSONB, nullable=False)
    score       = Column(Float, nullable=True)
    es_anomalia = Column(Boolean, nullable=True)
    created_at  = Column(DateTime, nullable=False, default=datetime.utcnow)



In [11]:
# Cómo se hace un INSERT y un SELECT con SQLAlchemy
print('INSERT (guardar inferencia):')
print("""
  orm = InferenciaORM(
      tipo="completa",
      input_json={"texto": "Paciente de 45 años..."},
      output_json={"extraccion": {...}, "anomalia": {...}},
      score=-0.31,
      es_anomalia=True,
  )
  db.add(orm)
  db.commit()       ← aquí se ejecuta el INSERT en PostgreSQL
""")

print('SELECT (listar con filtros):')
print("""
  q = db.query(InferenciaORM)
  q = q.filter(InferenciaORM.tipo == "completa")
  q = q.order_by(InferenciaORM.created_at.desc())
  rows = q.limit(10).all()
  
  SQL generado automáticamente:
  SELECT * FROM inferencias
  WHERE tipo = 'completa'
  ORDER BY created_at DESC
  LIMIT 10;
""")

INSERT (guardar inferencia):

  orm = InferenciaORM(
      tipo="completa",
      input_json={"texto": "Paciente de 45 años..."},
      output_json={"extraccion": {...}, "anomalia": {...}},
      score=-0.31,
      es_anomalia=True,
  )
  db.add(orm)
  db.commit()       ← aquí se ejecuta el INSERT en PostgreSQL

SELECT (listar con filtros):

  q = db.query(InferenciaORM)
  q = q.filter(InferenciaORM.tipo == "completa")
  q = q.order_by(InferenciaORM.created_at.desc())
  rows = q.limit(10).all()
  
  SQL generado automáticamente:
  SELECT * FROM inferencias
  WHERE tipo = 'completa'
  ORDER BY created_at DESC
  LIMIT 10;



---
# PARTE 6 — Docker y docker-compose

## ¿Por qué Docker?

Docker empaqueta tu aplicación + todas sus dependencias en un **contenedor**: una caja aislada que corre igual en cualquier computadora.

Sin Docker: "Funciona en mi máquina" — el clásico problema del desarrollador.
Con Docker: La misma imagen corre en tu laptop, en la laptop del profesor, en un servidor de producción.

## ¿Qué hace cada parte?

In [12]:
dockerfile_explicado = """
# ── Dockerfile — línea por línea ──────────────────────────────────────────────

FROM python:3.11-slim
# Base: imagen oficial de Python 3.11, versión "slim" (sin herramientas de compilación)
# Más pequeña que la estándar, suficiente para wheels binarios

ENV PYTHONDONTWRITEBYTECODE=1  PYTHONUNBUFFERED=1  MODEL_DIR=/app/models
# PYTHONDONTWRITEBYTECODE: no crear archivos .pyc (ahorra espacio)
# PYTHONUNBUFFERED: ver los logs en tiempo real en Docker

WORKDIR /app
# El directorio de trabajo dentro del contenedor

RUN apt-get install -y libpq-dev gcc
# psycopg2 necesita el cliente de PostgreSQL del sistema operativo

COPY requirements.txt .
RUN pip install -r requirements.txt
# Primero solo requirements.txt (antes del código)
# Docker cachea esta capa — si requirements.txt no cambia, no reinstala todo

RUN python -m spacy download es_core_news_sm
# Descarga el modelo de español de spaCy (12 MB)

COPY domain/ application/ infrastructure/  ./
# El código de la aplicación

EXPOSE 8000
CMD ["uvicorn", "infrastructure.api.main:app", "--host", "0.0.0.0", "--port", "8000"]
# Uvicorn es el servidor ASGI que corre FastAPI
"""
print(dockerfile_explicado)


# ── Dockerfile — línea por línea ──────────────────────────────────────────────

FROM python:3.11-slim
# Base: imagen oficial de Python 3.11, versión "slim" (sin herramientas de compilación)
# Más pequeña que la estándar, suficiente para wheels binarios

ENV PYTHONDONTWRITEBYTECODE=1  PYTHONUNBUFFERED=1  MODEL_DIR=/app/models
# PYTHONDONTWRITEBYTECODE: no crear archivos .pyc (ahorra espacio)
# PYTHONUNBUFFERED: ver los logs en tiempo real en Docker

WORKDIR /app
# El directorio de trabajo dentro del contenedor

RUN apt-get install -y libpq-dev gcc
# psycopg2 necesita el cliente de PostgreSQL del sistema operativo

COPY requirements.txt .
RUN pip install -r requirements.txt
# Primero solo requirements.txt (antes del código)
# Docker cachea esta capa — si requirements.txt no cambia, no reinstala todo

RUN python -m spacy download es_core_news_sm
# Descarga el modelo de español de spaCy (12 MB)

COPY domain/ application/ infrastructure/  ./
# El código de la aplicación

EXPOSE 8000
CMD 

In [13]:
compose_explicado = """
# ── docker-compose.yml — línea por línea ──────────────────────────────────────

services:

  db:
    image: postgres:16-alpine     # imagen oficial de PostgreSQL 16, versión alpine (mínima)
    environment:                  # variables de entorno del contenedor de Postgres
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}  # leída desde el .env
    volumes:
      - pgdata:/var/lib/postgresql/data  # datos persistentes (sobreviven docker-compose down)
    healthcheck:
      test: ["pg_isready ..."]    # comando que verifica si Postgres está listo
      interval: 5s                # revisar cada 5 segundos
      retries: 10                 # intentar hasta 10 veces antes de fallar

  api:
    build: .                      # construir imagen desde el Dockerfile en este directorio
    depends_on:
      db:
        condition: service_healthy  # NO arrancar la API hasta que Postgres pase su healthcheck
    environment:
      DATABASE_URL: postgresql://epiadmin:PASSWORD@db:5432/epidiagnostix
      # "db" es el nombre del servicio — Docker lo resuelve como hostname interno
    volumes:
      - ../models:/app/models:ro  # montar modelos entrenados (read-only)
    ports:
      - "8000:8000"               # puerto del host : puerto del contenedor

volumes:
  pgdata:        # volumen nombrado — Docker lo gestiona automáticamente
"""
print(compose_explicado)


# ── docker-compose.yml — línea por línea ──────────────────────────────────────

services:

  db:
    image: postgres:16-alpine     # imagen oficial de PostgreSQL 16, versión alpine (mínima)
    environment:                  # variables de entorno del contenedor de Postgres
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}  # leída desde el .env
    volumes:
      - pgdata:/var/lib/postgresql/data  # datos persistentes (sobreviven docker-compose down)
    healthcheck:
      test: ["pg_isready ..."]    # comando que verifica si Postgres está listo
      interval: 5s                # revisar cada 5 segundos
      retries: 10                 # intentar hasta 10 veces antes de fallar

  api:
    build: .                      # construir imagen desde el Dockerfile en este directorio
    depends_on:
      db:
        condition: service_healthy  # NO arrancar la API hasta que Postgres pase su healthcheck
    environment:
      DATABASE_URL: postgresql://epiadmin:PASSWORD@db:5432/epidiagnostix
 

---
# PARTE 7 — Cómo levantar y probar el sistema

In [14]:
pasos = """
=== PASOS PARA LEVANTAR EL SISTEMA ===

1. Instalar Docker Desktop (si no está instalado)
   https://www.docker.com/products/docker-desktop

2. Abrir terminal en la carpeta del microservicio:
   cd microservicio

3. Crear el archivo .env con la contraseña:
   cp .env.example .env
   # Editar .env y cambiar POSTGRES_PASSWORD

4. Construir y levantar (primera vez tarda ~3-5 min por descarga de dependencias):
   docker-compose up --build

5. Verificar que todo está corriendo:
   - En el navegador: http://localhost:8000/docs  (Swagger UI)
   - En terminal:     curl http://localhost:8000/health

6. Correr las pruebas de integración (en otra terminal):
   pytest tests/test_integration.py -v

7. Detener cuando termines:
   docker-compose down

=== PRUEBA MANUAL CON CURL ===

  # Extracción
  curl -X POST http://localhost:8000/extraccion \\
       -H "Content-Type: application/json" \\
       -d '{"texto": "Paciente masculino de 45 años, presión 125/82, glucosa 98 mg/dl"}'

  # Consulta completa
  curl -X POST http://localhost:8000/consulta-completa \\
       -H "Content-Type: application/json" \\
       -d '{"texto": "Consulta médica. Paciente masculino de 63 años. Presión 165/83 mmHg, glucosa 341 mg/dl, temperatura 39.0 grados, frecuencia cardiaca 113 lpm. Peso 68.8 kg, talla 149.4 cm. 1 día de evolución."}'
"""
print(pasos)


=== PASOS PARA LEVANTAR EL SISTEMA ===

1. Instalar Docker Desktop (si no está instalado)
   https://www.docker.com/products/docker-desktop

2. Abrir terminal en la carpeta del microservicio:
   cd microservicio

3. Crear el archivo .env con la contraseña:
   cp .env.example .env
   # Editar .env y cambiar POSTGRES_PASSWORD

4. Construir y levantar (primera vez tarda ~3-5 min por descarga de dependencias):
   docker-compose up --build

5. Verificar que todo está corriendo:
   - En el navegador: http://localhost:8000/docs  (Swagger UI)
   - En terminal:     curl http://localhost:8000/health

6. Correr las pruebas de integración (en otra terminal):
   pytest tests/test_integration.py -v

7. Detener cuando termines:
   docker-compose down

=== PRUEBA MANUAL CON CURL ===

  # Extracción
  curl -X POST http://localhost:8000/extraccion \
       -H "Content-Type: application/json" \
       -d '{"texto": "Paciente masculino de 45 años, presión 125/82, glucosa 98 mg/dl"}'

  # Consulta complet

---
# PARTE 8 — Resumen para la defensa

In [15]:
print('=' * 65)
print('  RESUMEN — Microservicio ML EpiDiagnostix-Mayab')
print('=' * 65)
print()
print('  Arquitectura: Hexagonal (Ports & Adapters)')
print('  Framework API: FastAPI + Uvicorn')
print('  Base de datos: PostgreSQL 16 (tabla inferencias con JSONB)')
print('  ORM: SQLAlchemy (create_all en startup — sin Alembic)')
print('  Contenedores: Docker Compose (2 servicios: api + db)')
print()
print('  Capas y responsabilidades:')
print('    domain/      → Entidades + Puertos (sin deps externas)')
print('    application/ → Casos de uso (orquesta adapters via puertos)')
print('    infrastructure/')
print('      api/       → FastAPI + Pydantic (entrada HTTP)')
print('      adapters/  → ExtractorAdapter (regex/spaCy)')
print('                   IsolationForestAdapter (sklearn joblib)')
print('                   PostgresRepository (SQLAlchemy)')
print('      db/        → ORM + Session')
print()
print('  Endpoints:')
print('    GET  /health              → Estado del servicio')
print('    POST /extraccion          → Texto → variables clínicas')
print('    POST /deteccion-anomalias → Variables → predicción IF')
print('    POST /consulta-completa   → Texto → extracción + IF')
print('    GET  /inferencias         → Historial en PostgreSQL')
print()
print('  Pruebas:')
print('    tests/test_integration.py → 7 tests pytest + httpx')
print('    postman/                  → Colección importable')
print()
print('  Documentación automática:')
print('    http://localhost:8000/docs   (Swagger UI)')
print('    http://localhost:8000/redoc  (ReDoc)')
print('=' * 65)

  RESUMEN — Microservicio ML EpiDiagnostix-Mayab

  Arquitectura: Hexagonal (Ports & Adapters)
  Framework API: FastAPI + Uvicorn
  Base de datos: PostgreSQL 16 (tabla inferencias con JSONB)
  ORM: SQLAlchemy (create_all en startup — sin Alembic)
  Contenedores: Docker Compose (2 servicios: api + db)

  Capas y responsabilidades:
    domain/      → Entidades + Puertos (sin deps externas)
    application/ → Casos de uso (orquesta adapters via puertos)
    infrastructure/
      api/       → FastAPI + Pydantic (entrada HTTP)
      adapters/  → ExtractorAdapter (regex/spaCy)
                   IsolationForestAdapter (sklearn joblib)
                   PostgresRepository (SQLAlchemy)
      db/        → ORM + Session

  Endpoints:
    GET  /health              → Estado del servicio
    POST /extraccion          → Texto → variables clínicas
    POST /deteccion-anomalias → Variables → predicción IF
    POST /consulta-completa   → Texto → extracción + IF
    GET  /inferencias         → Historia